# Large Run Step 6: Additional Plotting Driver

Purpose: create optional diagnostic figures from existing compact tables and bounded samples, without loading full QC or metric inventories in the notebook.

Outputs: waveform comparison figures, station/event maps, flexible metric plots, and selected diagnostics.


## Setup
Purpose: load the active config and shared notebook settings through package helpers.

Outputs: a compact run-context summary.


In [ ]:
from pathlib import Path
import runpy

# Make the local source checkout importable when running notebooks without an installed wheel.
_bootstrap = next(
    (
        path
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for path in (
            candidate / "_source_bootstrap.py",
            candidate / "docs" / "examples" / "_source_bootstrap.py",
        )
        if path.exists()
    ),
    None,
)
if _bootstrap is None:
    raise RuntimeError("Could not find docs/examples/_source_bootstrap.py.")
runpy.run_path(str(_bootstrap))["use_source_checkout"]()

from IPython.display import display

from spatial_vtk.config import (
    notebook_run_context,
    notebook_figure_settings,
    prepare_notebook_geospatial_environment,
    print_notebook_context,
)
from spatial_vtk.spatial import load_standard_additional_plotting_output_status

prepare_notebook_geospatial_environment()
context = notebook_run_context()
cfg = context.cfg
OVERWRITE = context.overwrite
QC_CHUNKSIZE = context.qc_chunksize
PREVIEW_ROWS = context.preview_rows

print_notebook_context(context)


## Resolve Plotting Inputs
Purpose: verify that comparison-eligible records and metric outputs are ready.

Outputs: status table only.


In [ ]:
plotting_outputs = load_standard_additional_plotting_output_status(cfg=cfg)

display(plotting_outputs.status_frame())


## Render Bounded Waveform Comparison
Purpose: load only a small comparison-eligible sample and render one waveform diagnostic figure.

Outputs: `event_trace_comparison` figure when `SVTK_MAKE_FIGURES=1`.


In [ ]:
WAVEFORM_FIGURE_SETTINGS = notebook_figure_settings("waveform")
waveform_result = plotting_outputs.write_waveform_comparison(
    WAVEFORM_FIGURE_SETTINGS,
    max_records=12,
    chunksize=QC_CHUNKSIZE,
    overwrite=OVERWRITE,
)
display(waveform_result.status_frame())


## Preview Metric Plot Inputs
Purpose: inspect only the first rows of the metric table selected for plotting.

Outputs: bounded preview table.


In [ ]:
plotting_outputs.display_metric_source_preview(nrows=PREVIEW_ROWS)


## Region Boxplot with Comparison Table
Purpose: reproduce the tutorial region-comparison boxplot with its bootstrap comparison table using a bounded metric sample.

Outputs: a region boxplot PNG plus the package-owned comparison-table preview/status for the selected metric sample.


In [ ]:
REGION_FIGURE_SETTINGS = notebook_figure_settings(
    "region",
    figure_subdir="metrics",
    default_metric="PGA",
    default_passband="2-3 sec",
    default_sidecar_rows=1000,
)
region_result = plotting_outputs.write_region_boxplot(
    REGION_FIGURE_SETTINGS,
    output_prefix="additional_region_boxplot",
    annotate_if_missing=False,
    overwrite=OVERWRITE,
)
display(region_result.comparison_frame())
display(region_result.status_frame())
